#### **This script evaluates the results for softunion and softunion null**

In [ ]:
import pandas as pd

In [ ]:
files = {
    'egypt_uae_082019_1': {
        'quantile_edges': 'egypt_uae_082019_1_tweets_softunion_merged_edges_0.95.csv',
        'pvalue_edges': 'egypt_uae_082019_1_tweets_softunion_null_merged_edges_0.05.csv',
        'campaign': 'egypt_uae_082019_1_tweets',
        
        # 'cohashtag': 'egypt_uae_082019_1_tweets_softunion_cohashtag_edges_0.95.csv',
        # 'coretweet': 'egypt_uae_082019_1_tweets_softunion_coretweet_edges_0.95.csv',
        # 'coretweetusers': 'egypt_uae_082019_1_tweets_softunion_coretweetusers_edges_0.95.csv',
        # 'coword': 'egypt_uae_082019_1_tweets_softunion_coword_edges_0.95.csv',
        
        # 'p_cohashtag': 'egypt_uae_082019_1_tweets_softunion_null_cohashtag_edges_0.95.csv',
        # 'p_coretweet': 'egypt_uae_082019_1_tweets_softunion_null_coretweet_edges_0.95.csv',
        # 'p_coretweetusers': 'egypt_uae_082019_1_tweets_softunion_null_coretweetusers_edges_0.95.csv',
        # 'p_coword': 'egypt_uae_082019_1_tweets_softunion_null_coword_edges_0.95.csv',
    },
    'thailand': {
        'quantile_edges': 'thailand_092020_tweets_softunion_merged_edges_0.95.csv',
        'pvalue_edges': 'thailand_092020_tweets_softunion_null_merged_edges_0.05.csv',
        'campaign': 'thailand_092020_tweets'
    },
     'iranian': {
        'quantile_edges': 'iranian_tweets_softunion_merged_edges_0.95.csv',
        'pvalue_edges': 'iranian_tweets_softunion_null_merged_edges_0.05.csv',
        'campaign': 'iranian_tweets'
    },
}

In [ ]:
path = '/N/slate/potem/project/coordinationz/Outputs/Tables/'

In [ ]:
io_path = '/N/project/INCAS/new_parse/'

#### **Check control nodes**

In [ ]:
df_control = pd.read_pickle('/N/project/INCAS/new_parse/control/thailand_092020_tweets_control.pkl.gz')

In [ ]:
df_control.columns

In [ ]:
df_control['userid'].nunique()

# **The number of nodes does not in edge file does not match in node file!!!**

In [ ]:
def load_org_data(campaign):

    print(campaign)

    io_path = '/N/project/INCAS/new_parse/'
    
    io_file = io_path + 'io/'+ files[campaign]['campaign'] + '_io.pkl.gz'
    control_file = io_path + 'control/' + files[campaign]['campaign'] + '_control.pkl.gz'

    print(io_file)
    print(control_file)
    import os

    if os.path.exists(io_file) == False:
        print(io_file, '   does not exist!!')
        return

    if os.path.exists(control_file) == False:
        print(control_file, '  does not exists!!!')
        return 
    
    df_org_io =  pd.read_pickle(io_file)
    df_org_io['y_org'] = 1
    
    df_org_control =  pd.read_pickle(control_file)
    df_org_control['y_org'] = 0
    
    df_org = pd.concat([df_org_io, df_org_control],
                       ignore_index=True
                      )
    print(df_org['y_org'].unique())

    df_org['userid'] = df_org['userid'].astype(str)

    return df_org

In [ ]:
campaign = 'egypt_uae_082019_1'

df_org = load_org_data(campaign)

In [ ]:
def individual_indicator(campaign, look_up, x):
    import os
    
    print(x)
    
    if x == 'merged':
        pvalue_path = path + files[campaign]['pvalue_edges']
        quant_path = path + files[campaign]['quantile_edges']
    else:
        pvalue_path = path + files[campaign]['p_' + x]
        quant_path = path + files[campaign][x]

        if os.path.exists(pvalue_path) == False:
            print(pvalue_path, '   does not exist!!')
            return
    
        if os.path.exists(quant_path) == False:
            print(quant_path, '  does not exists!!!')
            return 

    print(pvalue_path)
    print(quant_path)
    
    df_pvalue = pd.read_csv(pvalue_path, dtype={
         'source_id': str,
        'target_id': str
    })
    df_quant = pd.read_csv(quant_path,  dtype={
         'source_id': str,
        'target_id': str
    })

    # df_pvalue = df_pvalue.astype({
    #     'source_id': str,
    #     'target_id': str
    # })

    # df_quant = df_quant.astype({
    #     'source_id': str,
    #     'target_id': str
    # })

    return df_pvalue, df_quant

In [ ]:
df_pvalue, df_quant = individual_indicator(campaign, files, 'merged')

In [ ]:
def load_labels(df_org, df_edge):
    list_nodes = list(set(df_edge['source_id']).union(set(df_edge['target_id'])))
    df_label = pd.DataFrame(data=list_nodes,
                            columns=['user_id']
                           )

    df_label = df_label.astype({
        'user_id': str
    })

    print('not matching data:' , len(df_label.loc[~(df_label['user_id'].isin(df_org['userid']))]))

    #### **Considering all dataset**
    df_org_grp = df_org.groupby(['userid', 'y_org']).first().reset_index()

    print(df_org_grp['y_org'].unique())
    
    df_org_grp['y_pred'] = 1
    # df_org_grp.loc[df_org_grp['userid'].isin(df_label['user_id']), 'y_pred'] = 1
    df_org_grp.loc[~(df_org_grp['userid'].isin(df_label['user_id'])), 'y_pred'] = 0

    #### **Considering only the filtered dataset**
    df_label['y_pred'] = 1
    # df_label['y_org'] = 0

    df_label = df_label.merge(df_org_grp[['userid', 'y_org']],
                              left_on='user_id',
                              right_on='userid',
                              how='left'
                             )
    print(df_label['y_org'].unique())
    
    # df_label.loc[df_label['user_id'].isin(df_org_grp['userid']), 'y_org'] = df_org_grp['y_org']

    return df_org_grp, df_label

In [ ]:
df_org_grp, df_label = load_labels(df_org, df_pvalue)

In [ ]:
from sklearn.metrics import precision_score, recall_score, f1_score, classification_report

def metric(df,
           y_ground='y_org', 
           y_pred='y_pred'
          ):

    precision = precision_score(df[y_ground],
                                df[y_pred]
                               )
    print(f"Precision: {precision}")
    
    # Calculate recall
    recall = recall_score(df[y_ground],
                          df[y_pred]
                         )
    print(f"Recall: {recall}")
    
    # Calculate F1-score
    f1 = f1_score(df[y_ground],
                  df[y_pred]
                 )
    print(f"F1-Score: {f1}")

    report = classification_report(df[y_ground], df[y_pred])
    print(report)

In [ ]:
df_org_grp, df_label

In [ ]:
 metric(df_org_grp,
       y_ground='y_org', 
       y_pred='y_pred'
      )

In [ ]:
 metric(df_label,
       y_ground='y_org', 
       y_pred='y_pred'
      )

In [ ]:
def loop_indicator(indicator):
    indicators = ['merged', 'cohashtag', 'coretweet' , 'coretweetusers', 'coword']

    for x in indicators:
        df_pvalue, df_quant = individual_indicator(campaign, look_up, x)

    return df_pvalue, df_quant

In [ ]:
for campaign in files:
    # if campaign != 'thailand':
    #     continue

    print('Campaign :', campaign)
    df_org = load_org_data(campaign)
    df_pvalue, df_quant = individual_indicator(campaign, files, 'merged')
    df_org_grp, df_label = load_labels(df_org, df_pvalue)
    
    metric(df_org_grp,
           y_ground='y_org', 
           y_pred='y_pred'
          )

    print('************** Label data \n')

    print('*** Running for the labeled data \n\n***')
    df_org_grp, df_label = load_labels(df_org, df_quant)
    metric(df_org_grp,
           y_ground='y_org', 
           y_pred='y_pred'
          )

    print('\n \n New campaign')
    # break